# Clase 225 — Privacidad diferencial: Laplace, composición y DP-SGD

Sintético: dataset de salarios n=10_000. Requiere: `pip install numpy pandas scikit-learn`.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 10_000
B = 200_000   # cota superior salario (clipping para sensibilidad acotada)
salaries = np.clip(rng.lognormal(mean=10.8, sigma=0.6, size=n), 0, B)
df = pd.DataFrame({'salary': salaries})
print(f'n={n:,} | mean real={salaries.mean():,.0f} | >100k real={(salaries > 100_000).sum():,}')

## 1. Mecanismo de Laplace

`M(D) = f(D) + Lap(0, Δf/ε)`. Cumple ε-DP. Varianza teórica: `2·(Δf/ε)²`.

In [ ]:
def laplace_mechanism(value, sensitivity, epsilon, rng=rng):
    """Agrega ruido Laplace(0, sensitivity/epsilon) — cumple ε-DP."""
    scale = sensitivity / epsilon
    return value + rng.laplace(0.0, scale)

# Verificación empírica: 10k corridas, sensibilidad=1, ε=1 → var teórica = 2
samples = np.array([laplace_mechanism(0.0, 1.0, 1.0) for _ in range(10_000)])
print(f'varianza empírica: {samples.var():.4f} | teórica: {2 * (1/1)**2:.4f}')

## 2. Conteo privado: trade-off privacy-utility según ε

In [ ]:
true_count = (df.salary > 100_000).sum()
sens_count = 1.0   # cambiar 1 registro mueve el conteo a lo sumo en 1

trials = 500
rows = []
for eps in [0.1, 1.0, 10.0]:
    runs = np.array([laplace_mechanism(true_count, sens_count, eps) for _ in range(trials)])
    rows.append({'epsilon': eps, 'true': true_count, 'mean_estim': runs.mean().round(1),
                 'MAE': np.abs(runs - true_count).mean().round(2),
                 'noise_scale': sens_count / eps})
print(pd.DataFrame(rows).to_string(index=False))
print('\n→ ε=0.1: privacidad fuerte, ruido ~10. ε=10: utility alta, garantía vacía.')

## 3. Mean privado: clip + suma Laplace / n

In [ ]:
def private_mean(values, lower, upper, epsilon, rng=rng):
    """DP mean: clip a [lower, upper], suma + Laplace, divide por n."""
    clipped = np.clip(values, lower, upper)
    sens = upper - lower   # sensibilidad de la SUMA
    noisy_sum = laplace_mechanism(clipped.sum(), sens, epsilon, rng)
    return noisy_sum / len(values)

true_mean = df.salary.mean()
print(f'mean real: {true_mean:,.2f}\n')
for B_clip in [50_000, 200_000, 1_000_000]:
    estims = np.array([private_mean(df.salary.values, 0, B_clip, 1.0) for _ in range(200)])
    bias = estims.mean() - true_mean
    print(f'B={B_clip:>9,} | mean_DP≈{estims.mean():>12,.0f} | bias={bias:>+10,.0f} | std={estims.std():>8,.0f}')
print('\n→ B chico: bias por clipping. B grande: bias chico pero varianza alta.')

## 4. Histograma privado (ruido independiente por bin)

In [ ]:
bins = np.linspace(0, B, 11)   # 10 bins
true_hist, _ = np.histogram(df.salary, bins=bins)

# Cada bin: sensibilidad=1 (un registro cae a lo sumo en 1 bin).
# Ruido independiente por bin → consume ε una sola vez (parallel composition).
eps = 1.0
noisy_hist = np.array([laplace_mechanism(c, 1.0, eps) for c in true_hist]).round().astype(int)

out = pd.DataFrame({'bin_lo': bins[:-1].astype(int), 'bin_hi': bins[1:].astype(int),
                    'true': true_hist, f'DP_eps={eps}': noisy_hist,
                    'error': noisy_hist - true_hist})
print(out.to_string(index=False))

## 5. Composición básica: 10 queries × ε=0.1 → ε_total=1.0

In [ ]:
# Demo: 10 conteos sucesivos sobre el mismo dataset, cada uno con ε=0.1.
# El presupuesto total gastado es la SUMA (composición básica).
k = 10
eps_per = 0.1
eps_total = k * eps_per

true = (df.salary > 100_000).sum()
queries = np.array([laplace_mechanism(true, 1.0, eps_per) for _ in range(k)])
print(f'true count: {true}')
print(f'queries individuales (ε=0.1 c/u): {queries.round(1)}')
print(f'\npresupuesto total gastado: ε_total = k·ε = {k}·{eps_per} = {eps_total}')
print(f'comparación: una sola query con ε=1.0 → noise scale 1.0')
print(f'             10 queries ε=0.1 → cada una noise scale 10.0 (mucho peor)')
print('\n→ Lección: agrupá queries y gastá el budget de una vez si podés.')

## 6. Gaussiano vs Laplace: varianza para mismo (ε, δ)

In [ ]:
eps, delta, sens = 1.0, 1e-5, 1.0

# Laplace ε-DP puro: var = 2·(Δf/ε)²
var_lap = 2 * (sens / eps) ** 2

# Gaussiano (ε,δ)-DP: σ = sqrt(2 ln(1.25/δ)) · Δf / ε
sigma = np.sqrt(2 * np.log(1.25 / delta)) * sens / eps
var_gauss = sigma ** 2

print(f'ε={eps}, δ={delta}, Δf={sens}')
print(f'Laplace: var = {var_lap:.3f}, std = {np.sqrt(var_lap):.3f}')
print(f'Gaussiano: σ = {sigma:.3f}, var = {var_gauss:.3f}')
print('\n→ Una query: Laplace gana. Muchas queries: Gaussiano compone mejor (RDP / moments accountant).')

## 7. DP-SGD conceptual sobre regresión lineal

Por cada paso: (1) grad per-sample, (2) clip a norma `C`, (3) promedio batch, (4) ruido gaussiano `N(0, σ²C²)`.

In [ ]:
# Dataset sintético: y = X @ w_true + ruido
d = 5
w_true = np.array([1.0, -2.0, 0.5, 3.0, -1.0])
X = rng.normal(0, 1, size=(2000, d))
y = X @ w_true + rng.normal(0, 0.1, size=2000)

def sgd(X, y, dp=False, C=1.0, sigma=1.0, lr=0.01, epochs=20, batch=64, rng=rng):
    w = np.zeros(d)
    for _ in range(epochs):
        idx = rng.permutation(len(X))
        for s in range(0, len(X), batch):
            b = idx[s:s + batch]
            # Gradiente per-sample: g_i = (X_i·w - y_i) · X_i
            preds = X[b] @ w
            per_sample = ((preds - y[b])[:, None]) * X[b]   # (batch, d)
            if dp:
                # 1) clip per-sample a norma C
                norms = np.linalg.norm(per_sample, axis=1, keepdims=True)
                per_sample = per_sample * np.minimum(1.0, C / (norms + 1e-9))
                # 2) suma + ruido gaussiano N(0, σ²C²)
                noisy_sum = per_sample.sum(axis=0) + rng.normal(0, sigma * C, size=d)
                grad = noisy_sum / len(b)
            else:
                grad = per_sample.mean(axis=0)
            w = w - lr * grad
    return w

w_plain = sgd(X, y, dp=False)
w_dp = sgd(X, y, dp=True, C=1.0, sigma=1.0)

print(f'w_true:  {w_true}')
print(f'w_SGD:   {w_plain.round(3)}   ‖err‖={np.linalg.norm(w_plain - w_true):.4f}')
print(f'w_DPSGD: {w_dp.round(3)}   ‖err‖={np.linalg.norm(w_dp - w_true):.4f}')
print('\n→ DP-SGD converge cerca, con error proporcional al ruido. Trade-off privacy-utility en acción.')

## Ejercicio guiado

1. Sobre el dataset de salarios, publicá un "dashboard DP" con 5 estadísticas (count total, count >100k, mean, p50 aproximada por histograma, p90). Repartí `ε_total=1.0` entre las 5 queries y justificá la asignación.
2. Re-corré el ejercicio 3 (`private_mean`) con `n=100`, `n=1_000`, `n=10_000`. Verificá que la calidad mejora con `n` (el ruido es absoluto, no relativo).
3. En el DP-SGD: barré `σ ∈ {0.5, 1.0, 2.0, 5.0}` y graficá ‖w_dp − w_true‖. Identificá el codo.
4. Instalá `diffprivlib` y compará tu `private_mean` contra `diffprivlib.tools.mean(salaries, epsilon=1.0, bounds=(0, B))`.
5. Bonus: implementá composición avanzada `ε_total = sqrt(2k ln(1/δ))·ε + k·ε(e^ε − 1)` y compará vs básica para k=100.

## Conclusiones

- **(ε, δ)-DP** es la única noción formal: te dice cuánto puede aprender un atacante de tu registro.
- **Laplace** para ε-DP puro; **Gaussiano** para (ε, δ)-DP y deep learning (compose mejor con RDP).
- **Sensibilidad** se acota CLIPPEANDO la entrada: sin clip, no hay garantía.
- **Composición**: cada query gasta presupuesto — tracking obligatorio.
- **DP-SGD** (Abadi 2016) = clip per-sample + ruido gaussiano. Opacus / TF-Privacy lo hacen por vos en producción.